# 01 – Data Cleaning: CMS Medicare Geographic Variation (2014–2024)
**Input:** `data/raw/Medicare_GV_by_National_State_County_2024.csv`
**Output:** clean national, state and county tables in `data/clean/`

## 1. Load the raw data
FIPS code is read as text so county codes keep their leading zeros (e.g. `01001`).

In [2]:
import pandas as pd
raw_df = pd.read_csv(
    "../data/raw/Medicare_GV_by_National_State_County_2024.csv",
    dtype={"State and County FIPS Code":str})
raw_df.info()

/var/folders/47/y3rxj0mn3l74c5r0ythwsbg40000gp/T/ipykernel_46355/3419611489.py:2: DtypeWarning: Columns (222,223) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36994 entries, 0 to 36993
Columns: 246 entries, Year to PQI16 Lower Extremity Amputation Admission Rate (age 75+)
dtypes: int64(1), object(245)
memory usage: 69.4+ MB


## 2. Explore the raw data
- 36994 rows × 246 columns, years 2014–2024
- Geographic levels: National, State, County
- Age levels: All, <65, >=65 (age breakdowns exist only for national and state rows)

In [3]:
print(raw_df.shape)

(36994, 246)


In [4]:
print(raw_df.columns)

Index(['Year', 'Geographic Level', 'State or County',
       'State and County FIPS Code', 'Age Level',
       'Total Medicare Beneficiaries', 'Beneficiaries with Part A and Part B',
       'Original Medicare (OM) Beneficiaries', 'MA Beneficiaries',
       'MA Participation Rate',
       ...
       'PQI11 Bacterial Pneumonia Admission Rate (age < 65)',
       'PQI11 Bacterial Pneumonia Admission Rate (age 65-74)',
       'PQI11 Bacterial Pneumonia Admission Rate (age 75+)',
       'PQI12 UTI Admission Rate (age < 65)',
       'PQI12 UTI Admission Rate (age 65-74)',
       'PQI12 UTI Admission Rate (age 75+)',
       'PQI15 Asthma in Younger Adults Admission Rate (age < 40)',
       'PQI16 Lower Extremity Amputation Admission Rate (age < 65)',
       'PQI16 Lower Extremity Amputation Admission Rate (age 65-74)',
       'PQI16 Lower Extremity Amputation Admission Rate (age 75+)'],
      dtype='object', length=246)


In [5]:
display(raw_df.head())

,Year,Geographic Level,State or County,State and County FIPS Code,Age Level,Total Medicare Beneficiaries,Beneficiaries with Part A and Part B,Original Medicare (OM) Beneficiaries,MA Beneficiaries,MA Participation Rate,...,PQI11 Bacterial Pneumonia Admission Rate (age < 65),PQI11 Bacterial Pneumonia Admission Rate (age 65-74),PQI11 Bacterial Pneumonia Admission Rate (age 75+),PQI12 UTI Admission Rate (age < 65),PQI12 UTI Admission Rate (age 65-74),PQI12 UTI Admission Rate (age 75+),PQI15 Asthma in Younger Adults Admission Rate (age < 40),PQI16 Lower Extremity Amputation Admission Rate (age < 65),PQI16 Lower Extremity Amputation Admission Rate (age 65-74),PQI16 Lower Extremity Amputation Admission Rate (age 75+)
0,2014,National,National,NaN,All,"56,767,775","49,302,821","33,462,969","15,839,852",32.13%,...,686,499,"1,373",361,269,"1,120",286,147,50,47
1,2014,National,National,NaN,<65,"9,286,837","7,824,336","5,867,170","1,957,166",25.01%,...,686,NaN,NaN,361,NaN,NaN,286,147,NaN,NaN
2,2014,National,National,NaN,>=65,"47,480,938","41,478,485","27,595,799","13,882,686",33.47%,...,NaN,499,"1,373",NaN,269,"1,120",NaN,NaN,50,47
3,2014,State,AK,02,All,"84,573","71,860","71,383",477,0.66%,...,622,366,"1,271",181,152,703,*,130,*,72
4,2014,State,AK,02,<65,"13,294","11,641","11,578",63,0.54%,...,622,NaN,NaN,181,NaN,NaN,*,130,NaN,NaN


In [6]:
print(raw_df["Geographic Level"].value_counts())

Geographic Level
County      35146
State        1815
National       33
Name: count, dtype: int64


In [7]:
print(raw_df["Age Level"].value_counts())

Age Level
All     35762
<65       616
>=65      616
Name: count, dtype: int64


In [8]:
print(raw_df["Year"].unique())

[2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]


## 3. Select columns
Kept 19 columns needed for the research question: identifiers, spending,
hospital use, and context variables. [One line on why, e.g. per-capita measures
instead of totals so places of different sizes can be compared.]

In [9]:
keep_cols = [
    # Identifiers (needed for filtering and joining)
    "Year",
    "Geographic Level",
    "State or County",
    "State and County FIPS Code",
    "Age Level",

    # Context
    "Total Medicare Beneficiaries",
    "Original Medicare (OM) Beneficiaries",   # the group the spending figures likely describe; useful for weighting
    "MA Participation Rate",
    "Average Age",
    "Percent Female",
    "Percent Eligible for Medicaid",

    # Spending
    "Actual Per Capita Medicare Payment",
    "Standardized Per Capita Medicare Payment",
    "IP Per Capita Standardized Medicare Payment",   # option 2: subtract from total to get non-hospital spending
    "IP Standardized Medicare Payment as % of Total Standardized Medicare Payment",   # option 3: hospital share

    # Hospital use
    "IP Covered Stays Per 1,000 Beneficiaries",
    "% of Beneficiaries Using IP",
    "Hospital Readmission Rate",
    "Emergency Department Visits per 1,000 Beneficiaries",
]

In [10]:
trim_raw_df = raw_df[(keep_cols)].copy()

In [11]:
# Stay as text: labels, not amounts
text_cols = [
    "Geographic Level",
    "State or County",
    "State and County FIPS Code",
    "Age Level",
]

# Convert to numbers
num_cols = [
    # Context
    "Total Medicare Beneficiaries",
    "Original Medicare (OM) Beneficiaries",
    "MA Participation Rate",
    "Average Age",
    "Percent Female",
    "Percent Eligible for Medicaid",

    # Spending
    "Actual Per Capita Medicare Payment",
    "Standardized Per Capita Medicare Payment",
    "IP Per Capita Standardized Medicare Payment",
    "IP Standardized Medicare Payment as % of Total Standardized Medicare Payment",

    # Hospital use
    "IP Covered Stays Per 1,000 Beneficiaries",
    "% of Beneficiaries Using IP",
    "Hospital Readmission Rate",
    "Emergency Department Visits per 1,000 Beneficiaries",
]

## 4. Convert numeric columns
- Removed `$`, `,` and `%` and converted to numbers.
- CMS hides small values with `*`; these became missing (NaN).
- Check: missing values after conversion = number of `*` in the original, for every column.
- Percentages kept as 32.13 instead of 0.3213.

In [12]:
missing_before = trim_raw_df[num_cols].isna().sum()
print(missing_before)

Total Medicare Beneficiaries                                                    0
Original Medicare (OM) Beneficiaries                                            0
MA Participation Rate                                                           0
Average Age                                                                     0
Percent Female                                                                  0
Percent Eligible for Medicaid                                                   0
Actual Per Capita Medicare Payment                                              0
Standardized Per Capita Medicare Payment                                        0
IP Per Capita Standardized Medicare Payment                                     0
IP Standardized Medicare Payment as % of Total Standardized Medicare Payment    0
IP Covered Stays Per 1,000 Beneficiaries                                        0
% of Beneficiaries Using IP                                                     0
Hospital Readmis

In [13]:
clean_trim_raw_df = trim_raw_df.copy()

for col in num_cols:
    clean_trim_raw_df[col] = (
        trim_raw_df[col]
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("%","", regex=False))
    clean_trim_raw_df[col] = pd.to_numeric(clean_trim_raw_df[col], errors="coerce")



In [14]:
stars_before = (trim_raw_df[num_cols] == "*").sum()
stars_after = clean_trim_raw_df[num_cols].isna().sum()

print(stars_before == stars_after)

Total Medicare Beneficiaries                                                    True
Original Medicare (OM) Beneficiaries                                            True
MA Participation Rate                                                           True
Average Age                                                                     True
Percent Female                                                                  True
Percent Eligible for Medicaid                                                   True
Actual Per Capita Medicare Payment                                              True
Standardized Per Capita Medicare Payment                                        True
IP Per Capita Standardized Medicare Payment                                     True
IP Standardized Medicare Payment as % of Total Standardized Medicare Payment    True
IP Covered Stays Per 1,000 Beneficiaries                                        True
% of Beneficiaries Using IP                                      

In [15]:
clean_trim_raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36994 entries, 0 to 36993
Data columns (total 19 columns):
 #   Column                                                                        Non-Null Count  Dtype  
---  ------                                                                        --------------  -----  
 0   Year                                                                          36994 non-null  int64  
 1   Geographic Level                                                              36994 non-null  object 
 2   State or County                                                               36994 non-null  object 
 3   State and County FIPS Code                                                    36895 non-null  object 
 4   Age Level                                                                     36994 non-null  object 
 5   Total Medicare Beneficiaries                                                  36359 non-null  float64
 6   Original Medicare (OM) Benefic

In [16]:
clean_trim_raw_df.head()

,Year,Geographic Level,State or County,State and County FIPS Code,Age Level,Total Medicare Beneficiaries,Original Medicare (OM) Beneficiaries,MA Participation Rate,Average Age,Percent Female,Percent Eligible for Medicaid,Actual Per Capita Medicare Payment,Standardized Per Capita Medicare Payment,IP Per Capita Standardized Medicare Payment,IP Standardized Medicare Payment as % of Total Standardized Medicare Payment,"IP Covered Stays Per 1,000 Beneficiaries",% of Beneficiaries Using IP,Hospital Readmission Rate,"Emergency Department Visits per 1,000 Beneficiaries"
0,2014,National,National,NaN,All,56767775.0,33462969.0,32.13,71.0,55.05,21.48,9767.0,9191.0,2620.0,28.51,281.70,17.50,18.08,670.72
1,2014,National,National,NaN,<65,9286837.0,5867170.0,25.01,51.0,47.69,57.36,10732.0,10079.0,3285.0,32.60,342.60,18.18,22.69,1176.80
2,2014,National,National,NaN,>=65,47480938.0,27595799.0,33.47,76.0,56.61,13.85,9562.0,9002.0,2479.0,27.54,268.75,17.35,16.87,563.13
3,2014,State,AK,02,All,84573.0,71383.0,0.66,70.0,50.32,23.81,8602.0,6524.0,2114.0,32.40,194.81,13.60,13.71,555.31
4,2014,State,AK,02,<65,13294.0,11578.0,0.54,51.0,46.04,64.98,10573.0,8173.0,2583.0,31.60,239.07,14.58,17.92,1069.01


## 5. Split into national, state and county tables
- State and national: "All" ages only, to avoid counting people three times.
- County: only has "All" ages.

In [17]:
national_df = clean_trim_raw_df[
    (clean_trim_raw_df["Geographic Level"] == "National") & 
    (clean_trim_raw_df["Age Level"] == "All")].copy()

state_df = clean_trim_raw_df[
    (clean_trim_raw_df["Geographic Level"] == "State") & 
    (clean_trim_raw_df["Age Level"] == "All")].copy()

county_df = clean_trim_raw_df[
    (clean_trim_raw_df["Geographic Level"] == "County") & 
    (clean_trim_raw_df["Age Level"] == "All")].copy()

print(national_df.shape)
print(state_df.shape)
print(county_df.shape)

(11, 19)
(605, 19)
(35146, 19)


## 6. Missing values

In [18]:
missing_national = national_df[num_cols].isna().sum()
missing_state = state_df[num_cols].isna().sum()
missing_county = county_df[num_cols].isna().sum()

print(missing_national)
print(missing_state)
print(missing_county)


Total Medicare Beneficiaries                                                    0
Original Medicare (OM) Beneficiaries                                            0
MA Participation Rate                                                           0
Average Age                                                                     0
Percent Female                                                                  0
Percent Eligible for Medicaid                                                   0
Actual Per Capita Medicare Payment                                              0
Standardized Per Capita Medicare Payment                                        0
IP Per Capita Standardized Medicare Payment                                     0
IP Standardized Medicare Payment as % of Total Standardized Medicare Payment    0
IP Covered Stays Per 1,000 Beneficiaries                                        0
% of Beneficiaries Using IP                                                     0
Hospital Readmis

In [19]:
# Picked out the variable with the highest missing values

display(county_df["Total Medicare Beneficiaries"].describe())
county_df[county_df["Hospital Readmission Rate"].isna()]["Total Medicare Beneficiaries"].describe()

count    3.457700e+04
mean     1.994092e+04
std      5.449962e+04
min      1.200000e+01
25%      2.621000e+03
50%      6.109000e+03
75%      1.558100e+04
max      1.718250e+06
Name: Total Medicare Beneficiaries, dtype: float64

count    1231.000000
mean      474.240455
std       331.281153
min        12.000000
25%       231.000000
50%       405.000000
75%       607.000000
max      1976.000000
Name: Total Medicare Beneficiaries, dtype: float64

Missing values come from CMS suppressing small counts (*), mainly in small counties. We left them as missing rather than filling them in, since filling would invent values for small, mostly rural counties. Limitation: rural results lean toward larger rural counties.

In [20]:
print(national_df.duplicated(subset=["Geographic Level", "Year"]).sum())
print(state_df.duplicated(subset=["State or County", "Year"]).sum())
print(county_df.duplicated(subset=["State and County FIPS Code", "Year"]).sum())

0
0
0


In [21]:
print(state_df["Year"].value_counts().sort_index().reset_index())


    Year  count
0   2014     55
1   2015     55
2   2016     55
3   2017     55
4   2018     55
5   2019     55
6   2020     55
7   2021     55
8   2022     55
9   2023     55
10  2024     55


In [22]:
# Listing states to check how there are 55 states in the US 
state_df["State or County"].unique()

array(['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA',
       'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME',
       'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM',
       'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'PR', 'RI', 'SC', 'SD', 'TN',
       'TX', 'Territory', 'UT', 'VA', 'VI', 'VT', 'WA', 'WI', 'WV', 'WY',
       'ZZ'], dtype=object)

In [23]:
# checking if counties are wildly different across the years
print(county_df["Year"].value_counts().sort_index().reset_index())

    Year  count
0   2014   3194
1   2015   3195
2   2016   3194
3   2017   3194
4   2018   3194
5   2019   3194
6   2020   3194
7   2021   3196
8   2022   3197
9   2023   3197
10  2024   3197


In [24]:
drop_states = ["PR", "VI", "Territory", "ZZ"]
state_df = state_df[~state_df["State or County"].isin(drop_states)].copy()

In [25]:
kept_state_codes = state_df["State and County FIPS Code"].unique()
county_state_code = county_df["State and County FIPS Code"].str[:2]
county_df = county_df[county_state_code.isin(kept_state_codes)].copy()


Kept 50 states + DC. Dropped PR, VI, "Territory" and ZZ. Territories have a different Medicare structure, and "Territory"/ZZ are not real places, with all values hidden. Counties filtered to match.

in the ALL AGES tables, 33 rows had no FIPS: 11 national rows plus 22 state rows with ZZ and Territory (1 per year per territory). All were removed when removing Territory/ZZ; state and county tables now have complete FIPS codes

## 7. Sanity checks

In [26]:
print(state_df.shape)
print(county_df.shape)


(561, 19)
(35102, 19)


In [27]:
print(state_df["State and County FIPS Code"].str.len().value_counts())
print(county_df["State and County FIPS Code"].str.len().value_counts())


State and County FIPS Code
2    561
Name: count, dtype: int64
State and County FIPS Code
5    35102
Name: count, dtype: int64


Summary of cleaning decisions:
- kept "All ages"
- "*" left as missing
- 50 states + DC
- counties matched to those states
- percentages kept as 0–100
- ensured all Codes are 2 char for states and 5 char for county


## 8. Rural vs urban classification
- Source: USDA ERS Rural-Urban Continuum Codes 2023 [cite in report].
- Codes 1–3 = Urban (metro), 4–9 = Rural (nonmetro).
- Joined to county data by 5-digit FIPS (left join; row count unchanged).
- [X] county rows unmatched (mainly Connecticut, due to its 2022 switch to planning regions, plus [ ]). Excluded from rural/urban comparisons only.
- **Limitation:** 2023 classification applied to all years (2014–2024), so changes in a county's status over time are not captured.

In [28]:
codes_df = pd.read_csv("../data/raw/Ruralurbancontinuumcodes2023.csv", encoding="latin-1", dtype={"FIPS":str})

codes_df.info()

codes_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9703 entries, 0 to 9702
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   FIPS         9703 non-null   object
 1   State        9703 non-null   object
 2   County_Name  9703 non-null   object
 3   Attribute    9703 non-null   object
 4   Value        9703 non-null   object
dtypes: object(5)
memory usage: 379.1+ KB


,FIPS,State,County_Name,Attribute,Value
0,01001,AL,Autauga County,Population_2020,58805
1,01001,AL,Autauga County,RUCC_2023,2
2,01001,AL,Autauga County,Description,"Metro - Counties in metro areas of 250,000 to ..."
3,01003,AL,Baldwin County,Population_2020,231767
4,01003,AL,Baldwin County,RUCC_2023,3


In [29]:
rucc_codes_df = codes_df[codes_df["Attribute"] == "RUCC_2023"].copy()
print(rucc_codes_df.duplicated(subset="FIPS").sum())
print(rucc_codes_df["FIPS"].str.len().value_counts())

0
FIPS
5    3233
Name: count, dtype: int64


In [30]:
rucc_small_df = rucc_codes_df[["FIPS", "Value"]].copy()
rucc_small_df = rucc_small_df.rename(columns={"Value": "rucc_code"})
rucc_small_df["rucc_code"] = pd.to_numeric(rucc_small_df["rucc_code"], errors="coerce")

print(rucc_small_df["rucc_code"].value_counts().sort_index())      # expect only 1-9

rucc_code
1    483
2    398
3    371
4    204
5     81
6    385
7    248
8    468
9    595
Name: count, dtype: int64


In [31]:
# USDA convention: codes 1-3 = metro (urban), 4-9 = nonmetro (rural)
rucc_small_df["rural_urban"] = pd.cut(
    rucc_small_df["rucc_code"],
    bins=[0, 3, 9],
    labels=["Urban", "Rural"]
)
print(rucc_small_df["rural_urban"].value_counts())

rural_urban
Rural    1981
Urban    1252
Name: count, dtype: int64


In [32]:
rows_before = len(county_df)

county_df = pd.merge(
    county_df,
    rucc_small_df,
    left_on="State and County FIPS Code",
    right_on="FIPS",
    how="left"                     # keep every CMS county, even without a match
)
county_df = county_df.drop(columns=["FIPS"])   # duplicate code column from USDA

print("Rows before:", rows_before, "| after:", len(county_df))   # must be equal

Rows before: 35102 | after: 35102


In [33]:
unmatched = county_df[county_df["rural_urban"].isna()]
print("Unmatched county rows:", len(unmatched))
print(unmatched[["State or County", "State and County FIPS Code"]].drop_duplicates())

Unmatched county rows: 621
        State or County State and County FIPS Code
24           AK-UNKNOWN                      02000
25    AK-Valdez-Cordova                      02261
26      AK-Wade Hampton                      02270
93           AL-UNKNOWN                      01000
167          AR-UNKNOWN                      05000
...                 ...                        ...
2993         VT-UNKNOWN                      50000
3031         WA-UNKNOWN                      53000
3099         WI-UNKNOWN                      55000
3158         WV-UNKNOWN                      54000
3186         WY-UNKNOWN                      56000

[61 rows x 2 columns]


## 9. Rename and save

In [34]:
rename_map = {
    "Year": "year",
    "Geographic Level": "geo_level",
    "State or County": "name",
    "State and County FIPS Code": "fips",
    "Age Level": "age_level",
    "Total Medicare Beneficiaries": "benes_total",
    "Original Medicare (OM) Beneficiaries": "benes_om",
    "MA Participation Rate": "ma_rate_pct",
    "Average Age": "avg_age",
    "Percent Female": "female_pct",
    "Percent Eligible for Medicaid": "medicaid_pct",
    "Actual Per Capita Medicare Payment": "spend_pc_actual",
    "Standardized Per Capita Medicare Payment": "spend_pc_std",
    "IP Per Capita Standardized Medicare Payment": "ip_spend_pc_std",
    "IP Standardized Medicare Payment as % of Total Standardized Medicare Payment": "ip_share_pct",
    "IP Covered Stays Per 1,000 Beneficiaries": "ip_stays_per_1000",
    "% of Beneficiaries Using IP": "ip_users_pct",
    "Hospital Readmission Rate": "readmit_rate_pct",
    "Emergency Department Visits per 1,000 Beneficiaries": "ed_visits_per_1000",
}

national_df = national_df.rename(columns=rename_map)
state_df = state_df.rename(columns=rename_map)
county_df = county_df.rename(columns=rename_map)

# geo_level and age_level are the same in every row of each table, so drop them
national_df = national_df.drop(columns=["geo_level", "age_level"])
state_df = state_df.drop(columns=["geo_level", "age_level"])
county_df = county_df.drop(columns=["geo_level", "age_level"])

In [35]:
national_df.to_csv("../data/clean/national_clean.csv", index=False)
state_df.to_csv("../data/clean/state_clean.csv", index=False)
county_df.to_csv("../data/clean/county_clean.csv", index=False)

In [36]:
# Always read fips as text, here and in the analysis notebook
check_df = pd.read_csv("../data/clean/county_clean.csv", dtype={"fips": str})
check_df.info()
check_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35102 entries, 0 to 35101
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   year                35102 non-null  int64  
 1   name                35102 non-null  object 
 2   fips                35102 non-null  object 
 3   benes_total         34544 non-null  float64
 4   benes_om            34540 non-null  float64
 5   ma_rate_pct         33943 non-null  float64
 6   avg_age             34540 non-null  float64
 7   female_pct          34522 non-null  float64
 8   medicaid_pct        34378 non-null  float64
 9   spend_pc_actual     34540 non-null  float64
 10  spend_pc_std        34540 non-null  float64
 11  ip_spend_pc_std     34475 non-null  float64
 12  ip_share_pct        34475 non-null  float64
 13  ip_stays_per_1000   34475 non-null  float64
 14  ip_users_pct        34475 non-null  float64
 15  readmit_rate_pct    33315 non-null  float64
 16  ed_v

,year,name,fips,benes_total,benes_om,ma_rate_pct,avg_age,female_pct,medicaid_pct,spend_pc_actual,spend_pc_std,ip_spend_pc_std,ip_share_pct,ip_stays_per_1000,ip_users_pct,readmit_rate_pct,ed_visits_per_1000,rucc_code,rural_urban
0,2014,AK-Aleutians East,02013,168.0,117.0,NaN,71.0,43.59,23.08,6076.0,4577.0,1366.0,29.84,153.85,11.11,NaN,196.58,9.0,Rural
1,2014,AK-Aleutians West,02016,231.0,136.0,0.00,69.0,47.79,27.21,13251.0,9712.0,3305.0,34.03,205.88,16.91,NaN,198.53,9.0,Rural
2,2014,AK-Anchorage,02020,33247.0,27578.0,0.64,70.0,52.61,24.93,8500.0,6665.0,2215.0,33.24,194.83,13.57,13.85,597.32,2.0,Urban
3,2014,AK-Bethel,02050,1311.0,1059.0,NaN,71.0,49.58,57.22,10709.0,6839.0,2356.0,34.45,288.95,18.41,15.74,53.82,7.0,Rural
4,2014,AK-Bristol Bay,02060,122.0,104.0,0.00,72.0,42.31,17.31,9934.0,6648.0,3294.0,49.55,201.92,10.58,NaN,NaN,9.0,Rural


In [38]:
for column in raw_df.columns:
    if "risk" in column.lower() or "hcc" in column.lower():
        print(column)

**Nini’s comment:**
Ask whether high-spending places have sicker patients. The file includes an average beneficiary risk score, so check whether the relationship survives after accounting for it.

**Response:**
We checked the original 2014–2024 CMS dataset and found no column containing “risk” or “HCC.” The Average HCC Score (BENE_AVG_RISK_SCRE) appears in the earlier 2014–2022 CMS dictionary, but it is not listed in the current 2014–2024 dictionary.
We therefore cannot adjust our current spending–hospital-use analysis using this risk score. We will acknowledge that differences in beneficiary health needs may partly explain the observed relationships. Standardizing geographic payment rates does not account for those health differences.